In [1]:
import pandas as pd
import numpy as np


def allocate_proportionally(df, totals_row, weight_col,id_col,keep_only_new=True, exclude_cols=None,suffix="_allocated"):
    if isinstance(totals_row, pd.DataFrame):
        totals_row = totals_row.iloc[0]
    
    df = df.copy()

    total_weight = df[weight_col].sum()
    df["_weight"] = df[weight_col] / total_weight

    new_cols = []

    for col in totals_row.index:
        if col not in exclude_cols:
            new_col = f"{col}{suffix}"
            df[new_col] = df["_weight"] * totals_row[col]
            new_cols.append(new_col)

    if keep_only_new:
        df= df[[id_col, weight_col] + new_cols]

    return df

def normalize(column):
    return (column - column.min()) / (column.max() - column.min())



In [2]:
#creates and normalizes the age demographics dataframe

#load df
age_df = pd.read_csv(r"C:\Users\kiana\Documents\College\Graduate - ECU\DASC 6010\Final Project\age_data\age_demographics_ACSST5Y2024.S0101-Data.csv")

#clean df
age_df = age_df[age_df["Geographic Area Name"].str.contains("North Carolina", na=False)]
age_df = age_df.loc[:, ~age_df.columns.str.contains("Margin of Error")]
prefix = "Estimate!!Total!!Total population!!AGE!!"
cols_to_keep = age_df.columns[age_df.columns.str.contains(prefix)].tolist()
cols_to_keep = ["Geographic Area Name"] + cols_to_keep
age_df = age_df[cols_to_keep]
age_df.columns = [col.replace(prefix, "") for col in age_df.columns]

#normalize age columns
normalize_cols = age_df.drop(columns=["Geographic Area Name"])
normalized_age_df = normalize_cols.apply(normalize)
normalized_age_df["Geographic Area Name"] = age_df["Geographic Area Name"]
normalized_age_df = normalized_age_df[["Geographic Area Name"] + [col for col in normalized_age_df.columns if col != "Geographic Area Name"]]
normalized_age_df.columns = [
    normalized_age_df.columns[0]] + ['age_' + col for col in normalized_age_df.columns[1:]
]
normalized_age_df.head()


,Geographic Area Name,age_Under 5 years,age_5 to 9 years,age_10 to 14 years,age_15 to 19 years,age_20 to 24 years,age_25 to 29 years,age_30 to 34 years,age_35 to 39 years,age_40 to 44 years,age_45 to 49 years,age_50 to 54 years,age_55 to 59 years,age_60 to 64 years,age_65 to 69 years,age_70 to 74 years,age_75 to 79 years,age_80 to 84 years,age_85 years and over
1892,"Alamance County, North Carolina",0.134421,0.139738,0.143577,0.162622,0.152908,0.109662,0.115310,0.121212,0.114412,0.122983,0.139999,0.168671,0.167423,0.186453,0.186321,0.198388,0.228441,0.214740
1893,"Alexander County, North Carolina",0.022483,0.022277,0.024024,0.024751,0.022894,0.020343,0.021630,0.025361,0.020085,0.025490,0.027077,0.034141,0.038325,0.037283,0.047117,0.068730,0.041026,0.040063
1894,"Alleghany County, North Carolina",0.003702,0.005324,0.003714,0.005687,0.005089,0.002969,0.004946,0.004438,0.002341,0.004168,0.006324,0.007840,0.012937,0.021357,0.011304,0.022959,0.014454,0.018097
1895,"Anson County, North Carolina",0.013411,0.013377,0.014715,0.013327,0.016379,0.014805,0.014132,0.013504,0.012751,0.013214,0.014293,0.020038,0.019254,0.024580,0.019030,0.024734,0.032353,0.027277
1896,"Ashe County, North Carolina",0.012570,0.013054,0.014399,0.017741,0.014888,0.011713,0.012344,0.012417,0.015727,0.016349,0.018327,0.021543,0.037490,0.034078,0.049462,0.051612,0.040534,0.050882


In [3]:
#make population 18+ df to use for gun sales dataframe
age_cols_18plus = ["15 to 19 years","20 to 24 years","25 to 29 years","30 to 34 years","35 to 39 years",
    "40 to 44 years","45 to 49 years","50 to 54 years","55 to 59 years","60 to 64 years",
    "65 to 69 years","70 to 74 years","75 to 79 years","80 to 84 years","85 years and over"]

age_df_18plus=age_df.copy()

#assume 40% of 15-19 years column is 18+ and the rest is 15-17
age_df_18plus["Population_18plus"] = np.floor(0.4*age_df["15 to 19 years"] + age_df[age_cols_18plus[1:]].sum(axis=1))

age_df_18plus=age_df_18plus[["Geographic Area Name","Population_18plus"]]



#used gun safety survey infromation to determine the type of county and population that owned guns and then used this to estimate the number of gun owners in each county based on the 
#population and type of county. This is because North Carolina does not have a state law requiring the registration of firearms, meaning there is no official, publicly available database 
# listing registered firearms by county. While pistol purchase permits were formerly handled by local sheriffs, that requirement was eliminated in March 2023

#chrome-extension://efaidnbmnnnibpcajpcglclefindmkaj/https://connect.ncdot.gov/events/Documents/nc-county-classifications.pdf
#https://schs.dph.ncdhhs.gov/data/brfss/2021/nc/all/GunStat.html



data = [
    ('Alamance County, North Carolina', 'Suburban'),
    ('Alexander County, North Carolina', 'Rural'),
    ('Alleghany County, North Carolina', 'Rural'),
    ('Anson County, North Carolina', 'Rural'),
    ('Ashe County, North Carolina', 'Rural'),
    ('Avery County, North Carolina', 'Rural'),
    ('Beaufort County, North Carolina', 'Rural'),
    ('Bertie County, North Carolina', 'Rural'),
    ('Bladen County, North Carolina', 'Rural'),
    ('Brunswick County, North Carolina', 'Suburban'),
    ('Buncombe County, North Carolina', 'Urban'),
    ('Burke County, North Carolina', 'Rural'),
    ('Cabarrus County, North Carolina', 'Suburban'),
    ('Caldwell County, North Carolina', 'Rural'),
    ('Camden County, North Carolina', 'Rural'),
    ('Carteret County, North Carolina', 'Suburban'),
    ('Caswell County, North Carolina', 'Rural'),
    ('Catawba County, North Carolina', 'Suburban'),
    ('Chatham County, North Carolina', 'Rural'),
    ('Cherokee County, North Carolina', 'Rural'),
    ('Chowan County, North Carolina', 'Rural'),
    ('Clay County, North Carolina', 'Rural'),
    ('Cleveland County, North Carolina', 'Rural'),
    ('Columbus County, North Carolina', 'Rural'),
    ('Craven County, North Carolina', 'Rural'),
    ('Cumberland County, North Carolina', 'Urban'),
    ('Currituck County, North Carolina', 'Suburban'),
    ('Dare County, North Carolina', 'Suburban'),
    ('Davidson County, North Carolina', 'Rural'),
    ('Davie County, North Carolina', 'Rural'),
    ('Duplin County, North Carolina', 'Rural'),
    ('Durham County, North Carolina', 'Urban'),
    ('Edgecombe County, North Carolina', 'Rural'),
    ('Forsyth County, North Carolina', 'Urban'),
    ('Franklin County, North Carolina', 'Suburban'),
    ('Gaston County, North Carolina', 'Urban'),
    ('Gates County, North Carolina', 'Suburban'),
    ('Graham County, North Carolina', 'Rural'),
    ('Granville County, North Carolina', 'Rural'),
    ('Greene County, North Carolina', 'Rural'),
    ('Guilford County, North Carolina', 'Urban'),
    ('Halifax County, North Carolina', 'Rural'),
    ('Harnett County, North Carolina', 'Rural'),
    ('Haywood County, North Carolina', 'Suburban'),
    ('Henderson County, North Carolina', 'Suburban'),
    ('Hertford County, North Carolina', 'Rural'),
    ('Hoke County, North Carolina', 'Rural'),
    ('Hyde County, North Carolina', 'Rural'),
    ('Iredell County, North Carolina', 'Suburban'),
    ('Jackson County, North Carolina', 'Suburban'),
    ('Jones County, North Carolina', 'Rural'),
    ('Lee County, North Carolina', 'Rural'),
    ('Lenoir County, North Carolina', 'Rural'),
    ('Lincoln County, North Carolina', 'Suburban'),
    ('Macon County, North Carolina', 'Rural'),
    ('Madison County, North Carolina', 'Rural'),
    ('Martin County, North Carolina', 'Rural'),
    ('McDowell County, North Carolina', 'Rural'),
    ('Mecklenburg County, North Carolina', 'Urban'),
    ('Mitchell County, North Carolina', 'Rural'),
    ('Montgomery County, North Carolina', 'Rural'),
    ('Moore County, North Carolina', 'Suburban'),
    ('Nash County, North Carolina', 'Suburban'),
    ('New Hanover County, North Carolina', 'Urban'),
    ('Northampton County, North Carolina', 'Rural'),
    ('Onslow County, North Carolina', 'Suburban'),
    ('Orange County, North Carolina', 'Suburban'),
    ('Pamlico County, North Carolina', 'Rural'),
    ('Pasquotank County, North Carolina', 'Rural'),
    ('Pender County, North Carolina', 'Rural'),
    ('Perquimans County, North Carolina', 'Rural'),
    ('Person County, North Carolina', 'Rural'),
    ('Pitt County, North Carolina', 'Suburban'),
    ('Polk County, North Carolina', 'Rural'),
    ('Randolph County, North Carolina', 'Rural'),
    ('Richmond County, North Carolina', 'Rural'),
    ('Robeson County, North Carolina', 'Rural'),
    ('Rockingham County, North Carolina', 'Rural'),
    ('Rowan County, North Carolina', 'Suburban'),
    ('Rutherford County, North Carolina', 'Suburban'),
    ('Sampson County, North Carolina', 'Rural'),
    ('Scotland County, North Carolina', 'Rural'),
    ('Stanly County, North Carolina', 'Rural'),
    ('Stokes County, North Carolina', 'Rural'),
    ('Surry County, North Carolina', 'Rural'),
    ('Swain County, North Carolina', 'Suburban'),
    ('Transylvania County, North Carolina', 'Rural'),
    ('Tyrrell County, North Carolina', 'Rural'),
    ('Union County, North Carolina', 'Urban'),
    ('Vance County, North Carolina', 'Rural'),
    ('Wake County, North Carolina', 'Urban'),
    ('Warren County, North Carolina', 'Rural'),
    ('Washington County, North Carolina', 'Rural'),
    ('Wayne County, North Carolina', 'Rural'),
    ('Wilkes County, North Carolina', 'Rural'),
    ('Wilson County, North Carolina', 'Rural'),
    ('Yadkin County, North Carolina', 'Rural'),
    ('Yancey County, North Carolina', 'Rural'),
]


df = pd.DataFrame(data, columns=['Geographic Area Name', 'Classification'])


gun_ownership = {
    'Urban': 31.6,
    'Suburban': 41.6,
    'Rural': 49.3
}


df['Percentage Gun Ownership'] = df['Classification'].map(gun_ownership)

merged_df = pd.merge(df, age_df_18plus, on='Geographic Area Name', how='inner')
merged_df['Estimated Gun Owners'] = merged_df['Population_18plus'] * (merged_df['Percentage Gun Ownership'] / 100)
merged_df['Estimated Gun Owners'] = merged_df['Estimated Gun Owners'].round()
merged_df['normalized'] = normalize(merged_df['Estimated Gun Owners'])
merged_df = merged_df[['Geographic Area Name', 'normalized']].rename(
    columns={'normalized': 'gun_ownership'}
)
merged_df.head()

,Geographic Area Name,gun_ownership
0,"Alamance County, North Carolina",0.194241
1,"Alexander County, North Carolina",0.045889
2,"Alleghany County, North Carolina",0.011275
3,"Anson County, North Carolina",0.026042
4,"Ashe County, North Carolina",0.033908


In [4]:
#chrome-extension://efaidnbmnnnibpcajpcglclefindmkaj/https://www.remi.com/wp-content/uploads/2017/10/326-North-Carolina-General-Assembly-The-Economic-Impact-of-the-Military-on-North-Carolina-2013.pdf
#does not account for reserves and national guard this is just active duty

df_bases = pd.DataFrame([
    ('Camp Lejeune', 2, 20, 139, 31, 3707, 35625, 'Onslow County, North Carolina'),
    ('Cherry Point', 2, 0, 0, 757, 454, 7165, 'Craven County, North Carolina'),
    ('Fort Bragg', 2201, 48239, 0, 6709, 243, 50688, 'Cumberland County, North Carolina'),
    ('New River', 18, 2, 0, 5, 39, 6903, 'Onslow County, North Carolina'),
    ('Seymour Johnson', 4631, 0, 0, 6846, 0, 4633, 'Wayne County, North Carolina'),
    ('Unknown', 130, 721, 1433, 1357, 271, 3912, 'Unknown County, North Carolina')
], columns=['Base', 'Air Force', 'Army', 'Coast Guard', 'Marine Corps', 'Navy', 'Total', 'Geographic Area Name'])

unknown_row = df_bases[df_bases['Base'] == 'Unknown'].iloc[0]
fort_bragg_index = df_bases[df_bases['Base'] == 'Fort Bragg'].index[0]

for col in ['Air Force', 'Army', 'Coast Guard', 'Marine Corps', 'Navy', 'Total']:
    df_bases.at[fort_bragg_index, col] += unknown_row[col]

df_bases = df_bases[df_bases['Base'] != 'Unknown']

df_joined = pd.merge(df_bases, age_df[["Geographic Area Name"]], on='Geographic Area Name', how='right')

df_clean = df_joined[['Geographic Area Name', 'Total']]

df_clean['Total'] = df_clean['Total'].fillna(0)

df_clean['Total Forces Normalized'] = normalize(df_clean['Total'])

df_clean = df_clean[['Geographic Area Name', 'Total Forces Normalized']].rename(
    columns={'Total Forces Normalized': 'active_duty_forces'}
)

df_clean.head(30)

C:\Users\kiana\AppData\Local\Temp\ipykernel_16688\3960380182.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Total'] = df_clean['Total'].fillna(0)
C:\Users\kiana\AppData\Local\Temp\ipykernel_16688\3960380182.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Total Forces Normalized'] = normalize(df_clean['Total'])


,Geographic Area Name,active_duty_forces
0,"Alamance County, North Carolina",0.000000
1,"Alexander County, North Carolina",0.000000
2,"Alleghany County, North Carolina",0.000000
3,"Anson County, North Carolina",0.000000
4,"Ashe County, North Carolina",0.000000
5,"Avery County, North Carolina",0.000000
6,"Beaufort County, North Carolina",0.000000
7,"Bertie County, North Carolina",0.000000
8,"Bladen County, North Carolina",0.000000
9,"Brunswick County, North Carolina",0.000000


In [5]:
vet_df = pd.read_csv(r"C:\Users\kiana\Documents\College\Graduate - ECU\DASC 6010\Final Project\veteran_data.csv")
vet_df=vet_df.rename(columns={'20-Sep': 'veteran_population'})
vet_df ['veteran_population'] = normalize(vet_df['veteran_population'])
vet_df['County'] = vet_df['County'].apply(lambda x: f"{x} County, North Carolina")
vet_df = vet_df.drop(columns=['FIPS'])
vet_df = vet_df[['County','veteran_population']].rename(columns={'County': 'Geographic Area Name'})

vet_df.head()

,Geographic Area Name,veteran_population
0,"Alamance County, North Carolina",0.183176
1,"Alexander County, North Carolina",0.037727
2,"Alleghany County, North Carolina",0.011589
3,"Anson County, North Carolina",0.022714
4,"Ashe County, North Carolina",0.026213


In [12]:
clean_scf_df = pd.read_csv(r"c:\Users\kiana\Documents\College\Graduate - ECU\DASC 6010\Final Project\nc_soc_com_scores.csv")
clean_scf_df = clean_scf_df.rename(columns={'County': 'Geographic Area Name'})
clean_scf_df['Geographic Area Name'] = clean_scf_df['Geographic Area Name'].apply(lambda x: f"{x} County, North Carolina")
clean_scf_df=clean_scf_df.drop(columns=['vet_count', 'final_score'])
clean_scf_df.head()

,Geographic Area Name,harvest_yield,permit_density
0,"Randolph County, North Carolina",1.000000,0.276074
1,"Onslow County, North Carolina",0.484558,0.302291
2,"Moore County, North Carolina",0.679971,0.212627
3,"Union County, North Carolina",0.747336,0.431650
4,"Craven County, North Carolina",0.596171,0.212889


In [14]:
# Merge all dataframes with an outer join on 'Geographic Area Name'
merged_all = normalized_age_df.merge(merged_df, on='Geographic Area Name', how='outer') \
                              .merge(df_clean, on='Geographic Area Name', how='outer') \
                              .merge(vet_df, on='Geographic Area Name', how='outer') \
                              .merge(clean_scf_df, on='Geographic Area Name', how='outer')
merged_all['social_community_factors_score'] = merged_all.iloc[:, 1:].apply(lambda row: row.sum(), axis=1)
merged_all ['social_community_factors_score']= normalize(merged_all['social_community_factors_score'])
merged_all.head()

,Geographic Area Name,age_Under 5 years,age_5 to 9 years,age_10 to 14 years,age_15 to 19 years,age_20 to 24 years,age_25 to 29 years,age_30 to 34 years,age_35 to 39 years,age_40 to 44 years,...,age_70 to 74 years,age_75 to 79 years,age_80 to 84 years,age_85 years and over,gun_ownership,active_duty_forces,veteran_population,harvest_yield,permit_density,social_community_factors_score
0,"Alamance County, North Carolina",0.134421,0.139738,0.143577,0.162622,0.152908,0.109662,0.115310,0.121212,0.114412,...,0.186321,0.198388,0.228441,0.214740,0.194241,0.0,0.183176,0.511468,0.264145,0.183948
1,"Alexander County, North Carolina",0.022483,0.022277,0.024024,0.024751,0.022894,0.020343,0.021630,0.025361,0.020085,...,0.047117,0.068730,0.041026,0.040063,0.045889,0.0,0.037727,0.196496,0.076163,0.037988
2,"Alleghany County, North Carolina",0.003702,0.005324,0.003714,0.005687,0.005089,0.002969,0.004946,0.004438,0.002341,...,0.011304,0.022959,0.014454,0.018097,0.011275,0.0,0.011589,0.421347,0.019454,0.023681
3,"Anson County, North Carolina",0.013411,0.013377,0.014715,0.013327,0.016379,0.014805,0.014132,0.013504,0.012751,...,0.019030,0.024734,0.032353,0.027277,0.026042,0.0,0.022714,0.806032,0.036049,0.052034
4,"Ashe County, North Carolina",0.012570,0.013054,0.014399,0.017741,0.014888,0.011713,0.012344,0.012417,0.015727,...,0.049462,0.051612,0.040534,0.050882,0.033908,0.0,0.026213,0.460177,0.066436,0.043387


In [15]:
merged_all.to_csv('social_community_factors_output.csv', index=False)